# 03 — Integrar notícias, pregões e retornos

Material de apoio ao [artigo](00_artigo.ipynb). Usamos probabilidades já calculadas e o snapshot de preços para construir uma linha por empresa/pregão. Cada notícia aponta para o primeiro pregão estritamente posterior; a agregação ocorre depois desse mapeamento. Execute em ordem, com as saídas dos notebooks 01 e 02.

## 1. Configuração e entradas

Conferimos os hashes das entradas e a versão do calendário. Outra coleta pode alterar preços ajustados: investigue e documente qualquer diferença. `EXIGIR_HASHES_REFERENCIA=False` assume explicitamente entradas diferentes e registra seus hashes; não transforma a nova execução na referência anterior.

In [1]:
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import exchange_calendars as xcals
from IPython.display import display

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'docs/config_integracao.json').exists()), None)
assert RAIZ is not None, 'Abra o notebook dentro da pasta do projeto.'
CONFIG = json.loads((RAIZ / 'docs/config_integracao.json').read_text(encoding='utf-8'))
INICIO = pd.Timestamp(CONFIG['inicio_noticias'])
FIM = pd.Timestamp(CONFIG['fim_noticias'])
TICKERS = CONFIG['tickers']
EXIGIR_HASHES_REFERENCIA = True
def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
HASHES_ENTRADA = {rel: sha256(RAIZ / rel) for rel in CONFIG['hashes_referencia']}
if EXIGIR_HASHES_REFERENCIA:
    assert HASHES_ENTRADA == CONFIG['hashes_referencia'], (
        'As entradas diferem do snapshot fixado. Use os dados do pacote validado '
        'ou assuma uma nova coleta explicitamente e documente a mudança.')
assert xcals.__version__ == CONFIG['exchange_calendars'], 'Instale a versão fixada do calendário.'
PROCESSADOS = RAIZ / 'data/processed'
DOCS = RAIZ / 'docs'
for pasta in [PROCESSADOS, DOCS]:
    pasta.mkdir(parents=True, exist_ok=True)
HASH_REVISAO = {str(p.relative_to(RAIZ)): sha256(p)
                for p in (DOCS / 'revisao_manual').glob('*.csv')}
print('Projeto:', RAIZ.name, '| Python:', sys.version.split()[0])
print('Entradas conferidas; calendário XNYS:', xcals.__version__)

Projeto: reproducao | Python: 3.11.16
Entradas conferidas; calendário XNYS: 4.13.2


## 2. Validar notícias e preços

Conferimos datas, empresas, IDs, chaves, probabilidades e preços. Notícias compartilhadas entre empresas permanecem válidas. A junção não pode multiplicar linhas ou esconder duplicatas.

In [2]:
noticias = pd.read_csv(PROCESSADOS / 'noticias_com_sentimento.csv',
                       parse_dates=['news_date'], keep_default_na=False)
precos = pd.read_csv(PROCESSADOS / 'cotacoes_limpas_com_margem.csv',
                    parse_dates=['trade_date']).sort_values(['stock', 'trade_date']).reset_index(drop=True)
PROBS = ['p_positivo', 'p_negativo', 'p_neutro']
assert {'source_id', 'stock', 'news_date', 'headline', 'escore_sentimento', *PROBS} <= set(noticias)
assert {'stock', 'trade_date', 'adj_close', 'volume'} <= set(precos)
assert set(noticias.stock) == set(precos.stock) == set(TICKERS)
assert noticias.news_date.notna().all() and precos.trade_date.notna().all()
assert noticias.news_date.between(INICIO, FIM).all()
assert noticias.news_date.eq(noticias.news_date.dt.normalize()).all()
assert precos.trade_date.eq(precos.trade_date.dt.normalize()).all()
assert noticias.source_id.is_unique
assert not noticias.duplicated(['stock', 'news_date', 'headline']).any()
assert not precos.duplicated(['stock', 'trade_date']).any()
assert np.isfinite(precos.adj_close).all() and precos.adj_close.gt(0).all()
assert np.isfinite(noticias[PROBS].to_numpy()).all()
assert noticias[PROBS].ge(0).all().all() and noticias[PROBS].le(1).all().all()
assert np.allclose(noticias[PROBS].sum(axis=1), 1, atol=1e-6)
assert np.allclose(noticias.escore_sentimento, noticias.p_positivo - noticias.p_negativo, atol=1e-8)
print('Notícias:', len(noticias), '| Preços com margem:', len(precos))
display(precos.groupby('stock').agg(inicio=('trade_date', 'min'), fim=('trade_date', 'max'), linhas=('trade_date', 'size')))

Notícias: 6552 | Preços com margem: 3860


,inicio,fim,linhas
stock,,,
BAC,2016-12-15,2020-01-10,772
JNJ,2016-12-15,2020-01-10,772
KO,2016-12-15,2020-01-10,772
NFLX,2016-12-15,2020-01-10,772
NVDA,2016-12-15,2020-01-10,772


## 3. Conferir o calendário

Comparamos cada série ao calendário XNYS, detectando inclusive uma sessão ausente em todas as empresas. O fechamento extraordinário de 05/12/2018 é excluído; pregões reduzidos continuam válidos. Divergências são exportadas e interrompem a etapa. As fontes históricas constam da configuração.

In [3]:
calendario = xcals.get_calendar(CONFIG['calendario'],
                               start=precos.trade_date.min().strftime('%Y-%m-%d'),
                               end=precos.trade_date.max().strftime('%Y-%m-%d'))
esperadas = pd.DatetimeIndex(calendario.sessions).tz_localize(None).normalize()
diferencas = []
cobertura = []
for ticker in TICKERS:
    observadas = pd.DatetimeIndex(precos.loc[precos.stock.eq(ticker), 'trade_date'])
    for tipo, datas in [('ausente', esperadas.difference(observadas)),
                        ('extra', observadas.difference(esperadas))]:
        diferencas.extend({'stock': ticker, 'trade_date': d, 'problema': tipo} for d in datas)
    cobertura.append({'stock': ticker, 'sessoes_com_margem': len(observadas),
                      'sessoes_2017_2019': int(observadas.to_series().between(INICIO, FIM).sum())})
problemas_calendario = pd.DataFrame(diferencas, columns=['stock', 'trade_date', 'problema'])
problemas_calendario.to_csv(DOCS / 'auditoria_calendario.csv', index=False)
assert problemas_calendario.empty, 'Calendário divergente: consulte docs/auditoria_calendario.csv.'
assert pd.Timestamp('2018-12-05') not in esperadas
assert pd.Timestamp('2018-12-24') in esperadas
assert all(d['sessoes_2017_2019'] == 754 for d in cobertura)
display(pd.DataFrame(cobertura))
print('Nenhuma sessão ausente ou extra em relação ao XNYS.')

Nenhuma sessão ausente ou extra em relação ao XNYS.


,stock,sessoes_com_margem,sessoes_2017_2019
0,BAC,772,754
1,JNJ,772,754
2,KO,772,754
3,NFLX,772,754
4,NVDA,772,754


## 4. Calcular retornos

Usamos `100 × (adj_close / adj_close_anterior − 1)` na série completa, antes do recorte. Assim, o primeiro retorno de 2017 tem preço anterior. Retorno absoluto é o módulo do retorno percentual, uma medida diária de oscilação. Não interpolamos preços.

In [4]:
precos['trade_date_anterior'] = precos.groupby('stock')['trade_date'].shift(1)
precos['adj_close_anterior'] = precos.groupby('stock')['adj_close'].shift(1)
precos['retorno_pct'] = 100 * (precos.adj_close / precos.adj_close_anterior - 1)
precos['retorno_absoluto_pct'] = precos.retorno_pct.abs()
precos['retorno_no_recorte'] = precos.trade_date.between(INICIO, FIM)
assert precos.loc[precos.retorno_no_recorte, 'retorno_pct'].notna().all()
assert np.isfinite(precos.loc[precos.retorno_no_recorte, 'retorno_pct']).all()
display(precos.loc[precos.retorno_no_recorte].groupby('stock').head(1)[
    ['stock', 'trade_date_anterior', 'trade_date', 'adj_close_anterior', 'adj_close', 'retorno_pct']])

,stock,trade_date_anterior,trade_date,adj_close_anterior,adj_close,retorno_pct
11,BAC,2016-12-30,2017-01-03,17.743874,18.089125,1.945748
783,JNJ,2016-12-30,2017-01-03,88.387215,88.870514,0.546798
1555,KO,2016-12-30,2017-01-03,30.666000,30.917477,0.820049
2327,NFLX,2016-12-30,2017-01-03,12.380000,12.749000,2.980610
3099,NVDA,2016-12-30,2017-01-03,2.622557,2.506342,-4.431362


## 5. Associar cada data ao pregão posterior

A busca com `side='right'` exige `pregão anterior ≤ news_date < trade_date`. Preservamos ambas as datas e conferimos nove casos conhecidos de fins de semana, feriados e bordas. Notícias de 31/12/2019 recebem retorno de janeiro de 2020, conservado como margem fora do painel principal.

In [5]:
datas = noticias[['stock', 'news_date']].drop_duplicates().sort_values(['stock', 'news_date']).copy()
datas['trade_date'] = pd.NaT
for ticker in TICKERS:
    mask = datas.stock.eq(ticker)
    sessoes = precos.loc[precos.stock.eq(ticker), 'trade_date'].to_numpy(dtype='datetime64[ns]')
    pos = np.searchsorted(sessoes, datas.loc[mask, 'news_date'].to_numpy(dtype='datetime64[ns]'), side='right')
    assert (pos < len(sessoes)).all(), f'Sem pregão posterior: {ticker}'
    datas.loc[mask, 'trade_date'] = sessoes[pos]
mapa = noticias.merge(datas, on=['stock', 'news_date'], how='left', validate='many_to_one')
campos_preco = ['stock', 'trade_date', 'trade_date_anterior', 'adj_close',
                'adj_close_anterior', 'retorno_pct', 'retorno_absoluto_pct', 'retorno_no_recorte']
mapa = mapa.merge(precos[campos_preco], on=['stock', 'trade_date'], how='left',
                  validate='many_to_one', indicator='_juncao_preco')
assert len(mapa) == len(noticias)
assert mapa.source_id.is_unique
assert mapa._juncao_preco.eq('both').all()
assert mapa[['trade_date', 'trade_date_anterior', 'retorno_pct']].notna().all().all()
assert (mapa.trade_date > mapa.news_date).all()
assert (mapa.trade_date_anterior <= mapa.news_date).all()
mapa = mapa.drop(columns='_juncao_preco')
mapa['dias_ate_pregao'] = (mapa.trade_date - mapa.news_date).dt.days
mapa['retorno_fora_recorte'] = ~mapa.retorno_no_recorte
mapa = mapa.sort_values(['stock', 'trade_date', 'news_date', 'source_id']).reset_index(drop=True)
# Casos sintéticos com respostas conhecidas: dia útil, fim de semana, feriado e virada do ano.
casos = pd.DataFrame([
    ('2017-01-06', '2017-01-09', 'sexta-feira'),
    ('2017-01-07', '2017-01-09', 'sábado'),
    ('2017-01-08', '2017-01-09', 'domingo'),
    ('2017-07-03', '2017-07-05', 'véspera de feriado'),
    ('2017-07-04', '2017-07-05', 'feriado'),
    ('2018-12-04', '2018-12-06', 'fechamento extraordinário no dia seguinte'),
    ('2018-12-05', '2018-12-06', 'fechamento extraordinário'),
    ('2018-12-24', '2018-12-26', 'pregão reduzido seguido de Natal'),
    ('2019-12-31', '2020-01-02', 'virada do ano')],
    columns=['news_date', 'esperado', 'caso'])
for c in ['news_date', 'esperado']:
    casos[c] = pd.to_datetime(casos[c])
sessoes = esperadas.to_numpy(dtype='datetime64[ns]')
casos['obtido'] = sessoes[np.searchsorted(sessoes, casos.news_date.to_numpy(dtype='datetime64[ns]'), side='right')]
assert casos.obtido.eq(casos.esperado).all()
casos.to_csv(DOCS / 'casos_calendario.csv', index=False, date_format='%Y-%m-%d')
display(casos)
print('Notícias mapeadas:', len(mapa), '| Retornos fora de 2017–2019:', int(mapa.retorno_fora_recorte.sum()))

Notícias mapeadas: 6552 | Retornos fora de 2017–2019: 6


,news_date,esperado,caso,obtido
0,2017-01-06,2017-01-09,sexta-feira,2017-01-09
1,2017-01-07,2017-01-09,sábado,2017-01-09
2,2017-01-08,2017-01-09,domingo,2017-01-09
3,2017-07-03,2017-07-05,véspera de feriado,2017-07-05
4,2017-07-04,2017-07-05,feriado,2017-07-05
5,2018-12-04,2018-12-06,fechamento extraordinário no dia seguinte,2018-12-06
6,2018-12-05,2018-12-06,fechamento extraordinário,2018-12-06
7,2018-12-24,2018-12-26,pregão reduzido seguido de Natal,2018-12-26
8,2019-12-31,2020-01-02,virada do ano,2020-01-02


## 6. Agregar por empresa/pregão

Todas as manchetes mapeadas contribuem com o mesmo peso para as probabilidades médias. Calculamos escore, módulo do escore, classe e contagens. Não usamos média simples de médias diárias com quantidades diferentes. Empates são explícitos. Datas civis distintas e notícias compartilhadas continuam rastreáveis.

In [6]:
por_pregao = mapa.groupby(['stock', 'trade_date'], as_index=False).agg(
    n_noticias=('source_id', 'size'), n_datas_noticias=('news_date', 'nunique'),
    primeira_news_date=('news_date', 'min'), ultima_news_date=('news_date', 'max'),
    p_positivo_medio=('p_positivo', 'mean'), p_negativo_medio=('p_negativo', 'mean'),
    p_neutro_medio=('p_neutro', 'mean'),
    n_compartilhadas=('compartilhada_entre_empresas', 'sum'))
por_pregao['escore_sentimento'] = por_pregao.p_positivo_medio - por_pregao.p_negativo_medio
por_pregao['intensidade_sentimento'] = por_pregao.escore_sentimento.abs()
medias = por_pregao[['p_positivo_medio', 'p_negativo_medio', 'p_neutro_medio']].to_numpy()
empates = np.isclose(medias, medias.max(axis=1, keepdims=True),
                     atol=CONFIG['tolerancia_empate'], rtol=0).sum(axis=1) > 1
classes = np.array(['positivo', 'negativo', 'neutro'])[medias.argmax(axis=1)]
por_pregao['classe_sentimento'] = np.where(empates, 'empate', classes)
por_pregao['empate_probabilidades'] = empates
por_pregao['ao_menos_3_noticias'] = por_pregao.n_noticias.ge(3)
por_pregao = por_pregao.merge(precos[campos_preco], on=['stock', 'trade_date'],
                             how='left', validate='one_to_one')
# Janela completa exige que todo intervalo [pregão anterior, pregão alvo) esteja no recorte de notícias.
por_pregao['janela_noticias_completa'] = ((por_pregao.trade_date_anterior >= INICIO) &
                                         (por_pregao.trade_date <= FIM + pd.Timedelta(days=1)))
por_pregao['elegivel_analise_principal'] = por_pregao.retorno_no_recorte & por_pregao.janela_noticias_completa
assert por_pregao.n_noticias.sum() == len(noticias)
assert not por_pregao.duplicated(['stock', 'trade_date']).any()
assert np.allclose(medias.sum(axis=1), 1, atol=1e-6)
display(por_pregao.groupby('stock').agg(pregoes_com_noticias=('trade_date', 'size'),
         noticias=('n_noticias', 'sum'), varias_datas=('n_datas_noticias', lambda x: int(x.gt(1).sum()))))
print('Unidades empresa/pregão:', len(por_pregao))

Unidades empresa/pregão: 2392


,pregoes_com_noticias,noticias,varias_datas
stock,,,
BAC,438,871,13
JNJ,446,1046,23
KO,335,644,11
NFLX,629,2344,75
NVDA,544,1647,30


## 7. Construir o painel

Incluímos todos os pregões de 2017–2019. Sem notícias, apenas contagens recebem zero; probabilidades, classe e escore ficam ausentes. Janelas incompletas nas bordas são sinalizadas. A comparação principal exige `tem_noticias & elegivel_analise_principal`, preservando os casos excluídos para conferência.

In [7]:
campos_sentimento = [c for c in por_pregao.columns if c not in campos_preco or c in ['stock', 'trade_date']]
painel = precos.loc[precos.retorno_no_recorte].merge(
    por_pregao[campos_sentimento], on=['stock', 'trade_date'], how='left', validate='one_to_one')
painel['tem_noticias'] = painel.n_noticias.notna()
for c in ['n_noticias', 'n_datas_noticias', 'n_compartilhadas']:
    painel[c] = painel[c].fillna(0).astype(int)
for c in ['empate_probabilidades', 'ao_menos_3_noticias']:
    painel[c] = painel[c].fillna(False).astype(bool)
painel['janela_noticias_completa'] = ((painel.trade_date_anterior >= INICIO) &
                                      (painel.trade_date <= FIM + pd.Timedelta(days=1)))
painel['elegivel_analise_principal'] = painel.retorno_no_recorte & painel.janela_noticias_completa
painel['situacao_noticias'] = np.where(painel.tem_noticias, 'com_noticias', 'sem_noticias_no_recorte')
sem_noticias = ~painel.tem_noticias
assert painel.loc[sem_noticias, ['p_positivo_medio', 'p_negativo_medio', 'p_neutro_medio',
                               'escore_sentimento', 'intensidade_sentimento', 'classe_sentimento']].isna().all().all()
assert not painel.duplicated(['stock', 'trade_date']).any()
assert len(painel) == 754 * len(TICKERS)
assert painel.retorno_pct.notna().all()
resumo = painel.groupby('stock', as_index=False).agg(
    pregoes=('trade_date', 'size'), com_noticias=('tem_noticias', 'sum'),
    noticias=('n_noticias', 'sum'), janelas_completas=('janela_noticias_completa', 'sum'))
resumo['sem_noticias_no_recorte'] = resumo.pregoes - resumo.com_noticias
display(resumo)
print('Com notícias e elegíveis para análise principal:',
      int((painel.tem_noticias & painel.elegivel_analise_principal).sum()))

Com notícias e elegíveis para análise principal: 2387


,stock,pregoes,com_noticias,noticias,janelas_completas,sem_noticias_no_recorte
0,BAC,754,438,871,753,316
1,JNJ,754,446,1046,753,308
2,KO,754,335,644,753,419
3,NFLX,754,628,2341,753,126
4,NVDA,754,543,1644,753,211


## 8. Auditar e exportar

Conferimos a preservação dos IDs, das entradas e da revisão humana, e verificamos as médias por uma construção alternativa. Exportamos painel, mapeamento, agregados e relatórios, com releitura e manifesto. A tabela por manchete serve à rastreabilidade; retornos não são tratados como observações independentes por notícia.

In [8]:
assert set(mapa.source_id) == set(noticias.source_id)
# Verificação por construção independente, ponderando cada probabilidade por sua contagem.
totais = mapa.groupby(['stock', 'trade_date'])[PROBS].sum().sort_index()
agregado = por_pregao.set_index(['stock', 'trade_date']).sort_index()
for original, medio in zip(PROBS, ['p_positivo_medio', 'p_negativo_medio', 'p_neutro_medio']):
    assert np.allclose(agregado[medio], totais[original] / agregado.n_noticias, atol=1e-12)
arquivos = {
    'data/processed/noticias_pregao_retornos.csv': mapa,
    'data/processed/sentimento_por_pregao.csv': por_pregao,
    'data/processed/base_integrada_pregoes.csv': painel,
    'data/processed/retornos_cotacoes.csv': precos,
    'docs/mapa_datas_pregoes.csv': datas,
    'docs/resumo_integracao_por_empresa.csv': resumo,
    'docs/noticias_retorno_fora_recorte.csv': mapa.loc[mapa.retorno_fora_recorte],
    'docs/grupos_varias_datas.csv': por_pregao.loc[por_pregao.n_datas_noticias.gt(1)]}
hashes_saida = {}
for rel, tabela in arquivos.items():
    tabela.to_csv(RAIZ / rel, index=False, encoding='utf-8', date_format='%Y-%m-%d', float_format='%.12g')
    relida = pd.read_csv(RAIZ / rel)
    assert len(relida) == len(tabela) and list(relida) == list(tabela), rel
    hashes_saida[rel] = sha256(RAIZ / rel)
painel_relido = pd.read_csv(PROCESSADOS / 'base_integrada_pregoes.csv', parse_dates=['trade_date'])
assert not painel_relido.duplicated(['stock', 'trade_date']).any()
assert painel_relido.loc[~painel_relido.tem_noticias, 'escore_sentimento'].isna().all()
assert HASHES_ENTRADA == {rel: sha256(RAIZ / rel) for rel in HASHES_ENTRADA}
assert HASH_REVISAO == {rel: sha256(RAIZ / rel) for rel in HASH_REVISAO}
manifesto = {
    'etapa': CONFIG['etapa'], 'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
    'inicio_noticias': str(INICIO.date()), 'fim_noticias': str(FIM.date()),
    'unidade_analitica': CONFIG['unidade_analitica'], 'regra_alinhamento': CONFIG['regra_alinhamento'],
    'retorno': CONFIG['retorno'], 'calendario': CONFIG['calendario'],
    'config_sha256': sha256(DOCS / 'config_integracao.json'),
    'versoes': {p: version(p) for p in ['pandas', 'numpy', 'exchange-calendars', 'pyluach', 'toolz']},
    'python': sys.version.split()[0], 'hashes_entrada': HASHES_ENTRADA,
    'entradas_iguais_referencia': HASHES_ENTRADA == CONFIG['hashes_referencia'],
    'snapshot_precos': CONFIG['snapshot_precos'],
    'contagens': {
        'noticias': len(noticias), 'pares_empresa_data_civil': len(datas),
        'pares_empresa_pregao_com_noticias': len(por_pregao),
        'grupos_com_varias_datas': int(por_pregao.n_datas_noticias.gt(1).sum()),
        'painel_2017_2019': len(painel), 'pregoes_com_noticias_2017_2019': int(painel.tem_noticias.sum()),
        'pregoes_sem_noticias_no_recorte': int((~painel.tem_noticias).sum()),
        'janelas_incompletas_painel': int((~painel.janela_noticias_completa).sum()),
        'com_noticias_e_elegiveis_analise_principal': int((painel.tem_noticias & painel.elegivel_analise_principal).sum()),
        'noticias_retorno_fora_recorte': int(mapa.retorno_fora_recorte.sum()),
        'grupos_retorno_fora_recorte': int((~por_pregao.retorno_no_recorte).sum()),
        'noticias_sem_pregao_ou_retorno': int(mapa[['trade_date', 'retorno_pct']].isna().any(axis=1).sum()),
        'diferencas_calendario': len(problemas_calendario),
        'casos_sinteticos_conferidos': len(casos),
        'empates_sentimento_pregao': int(por_pregao.empate_probabilidades.sum())},
    'fontes_calendario': CONFIG['fontes_calendario'],
    'hashes_saida': hashes_saida,
    'preservacao_entradas_e_revisao_humana': True,
    'estado': {'integracao': 'concluida', 'continuidade': '00_artigo.ipynb'},
    'limitacoes': [
        'Alinhamento por data civil, sem horário, não estabelece causalidade.',
        'Ausência no corpus não representa ausência de notícias no mundo.',
        'Primeiro pregão de 2017 e extensão em janeiro de 2020 têm janela de notícias incompleta.',
        'Manchetes compartilhadas e retornos seriados implicam dependência entre unidades.',
        'Revisão humana dirigida do FinBERT não estima acurácia geral.']}
(DOCS / 'manifesto_integracao.json').write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
display(pd.Series(manifesto['contagens'], name='valor').to_frame())
print('Integração concluída; arquivos exportados e relidos; entradas e revisão preservadas.')

Integração concluída; arquivos exportados e relidos; entradas e revisão preservadas.


,valor
noticias,6552
pares_empresa_data_civil,2569
pares_empresa_pregao_com_noticias,2392
grupos_com_varias_datas,152
painel_2017_2019,3770
pregoes_com_noticias_2017_2019,2390
pregoes_sem_noticias_no_recorte,1380
janelas_incompletas_painel,5
com_noticias_e_elegiveis_analise_principal,2387
noticias_retorno_fora_recorte,6


## 9. Continuidade

O [artigo](00_artigo.ipynb) usa o painel para as perguntas 4–7 e os registros por data civil para as perguntas 1–3. O filtro de três notícias é aplicado por empresa/pregão. Se manchetes forem excluídas, será necessário reagrupar antes de reaplicar esse filtro. Veja as regras e bordas em [integração](../docs/integracao.md).